# Pick & Go — Entrainement YOLOv8 (Google Colab)

**Instructions :**
1. `Execution > Modifier le type d'execution` -> choisir **GPU T4**
2. Executer toutes les cellules dans l'ordre (`Ctrl+F9`)
3. Telecharger `best.pt` a la fin

> Si OOM (crash memoire) : changer `batch=16` -> `batch=8` dans la cellule 6.

In [ ]:
# 0. Verification GPU + espace disque
import subprocess, psutil

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'AUCUN GPU — change le type d execution!')

disk = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print('Disque:', disk.stdout.split('\n')[1])

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f} GB dispo / {ram.total/1e9:.1f} GB total')

In [ ]:
# 1. Installation des dependances
!pip install ultralytics icrawler -q
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
# 2. Catalogue des produits
PRODUCTS = {
    'Bouteille_Eau': {'nom': 'Bouteille Eau',  'prix': 300},
    'Cahier':        {'nom': 'Cahier',          'prix': 500},
    'Stylo':         {'nom': 'Stylo',           'prix': 200},
    'Gazelle':       {'nom': 'Biere Gazelle',   'prix': 800},
    'Choco_Pain':    {'nom': 'Choco Pain',      'prix': 250},
    'Bissap':        {'nom': 'Jus Bissap',      'prix': 400},
    'Riz':           {'nom': 'Sac de Riz',      'prix': 1500},
    'Huile':         {'nom': 'Huile',           'prix': 2000},
}

SEARCH_QUERIES = {
    'Bouteille_Eau': 'bouteille eau minerale plastique',
    'Cahier':        'cahier scolaire spirale',
    'Stylo':         'stylo bille bleu noir',
    'Gazelle':       'gazelle biere bouteille senegal',
    'Choco_Pain':    'pain chocolat viennoiserie',
    'Bissap':        'bissap jus hibiscus sachet',
    'Riz':           'sac riz 5kg senegal',
    'Huile':         'huile cuisine bouteille litre',
}

IMAGES_PER_CLASS = 80
TRAIN_RATIO = 0.8
print(f'Catalogue: {len(PRODUCTS)} classes')
for k, v in PRODUCTS.items():
    print(f'  {k}: {v["prix"]} FCFA')

In [ ]:
# 3. Collecte des images via Bing
import os, shutil, random
from pathlib import Path
from icrawler.builtin import BingImageCrawler

RAW = Path('data/raw')

for class_name in PRODUCTS:
    save_dir = RAW / class_name
    save_dir.mkdir(parents=True, exist_ok=True)
    existing = list(save_dir.glob('*.*'))
    if len(existing) >= IMAGES_PER_CLASS:
        print(f'[OK] {class_name}: {len(existing)} images')
        continue
    print(f'[DL] {class_name}...')
    try:
        crawler = BingImageCrawler(
            feeder_threads=1, parser_threads=1, downloader_threads=2,
            storage={'root_dir': str(save_dir)}
        )
        crawler.crawl(keyword=SEARCH_QUERIES[class_name],
                      max_num=IMAGES_PER_CLASS, min_size=(80, 80))
    except Exception as e:
        print(f'  Erreur: {e}')
    count = len(list(save_dir.glob('*.*')))
    print(f'  -> {count} images')

print('\nResume:')
for cn in PRODUCTS:
    n = len(list((RAW / cn).glob('*.*')))
    print(f'  {"OK" if n >= 20 else "WARN"} {cn}: {n} images')

In [ ]:
# 4. Annotation YOLO + split train/val
from pathlib import Path
import shutil, random

VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

for d in ['data/train/images', 'data/train/labels',
          'data/val/images',   'data/val/labels']:
    Path(d).mkdir(parents=True, exist_ok=True)

class_names = list(PRODUCTS.keys())
total_train = total_val = 0

for class_id, class_name in enumerate(class_names):
    images = [f for f in (Path('data/raw') / class_name).iterdir()
              if f.suffix.lower() in VALID_EXT]
    if not images:
        print(f'  SKIP {class_name}: 0 image')
        continue
    random.shuffle(images)
    split = max(1, int(len(images) * TRAIN_RATIO))
    for phase, subset in [('train', images[:split]), ('val', images[split:])]:
        for img in subset:
            stem = f'{class_name}_{img.stem}'
            shutil.copy2(img, f'data/{phase}/images/{stem}{img.suffix}')
            Path(f'data/{phase}/labels/{stem}.txt').write_text(
                f'{class_id} 0.5 0.5 0.9 0.9\n')
    t, v = split, len(images) - split
    print(f'  {class_name:20s}: {t} train | {v} val')
    total_train += t; total_val += v

print(f'\nDataset: {total_train} train | {total_val} val')

In [ ]:
# 5. Creation du data.yaml
yaml = 'path: /content\ntrain: data/train/images\nval:   data/val/images\n\n'
yaml += f'nc: {len(PRODUCTS)}\nnames:\n'
for name in PRODUCTS:
    yaml += f'  - {name}\n'

with open('data.yaml', 'w') as f:
    f.write(yaml)
print(yaml)

In [ ]:
# 6. Entrainement YOLOv8 sur GPU
import torch
from ultralytics import YOLO

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

model = YOLO('yolov8n.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # -> mettre 8 si crash OOM
    device=0 if torch.cuda.is_available() else 'cpu',
    project='models',
    name='pick_and_go',
    exist_ok=True,
    patience=10,
)

print('Entrainement termine!')

In [ ]:
# 7. Localisation du modele entraine
import subprocess
from pathlib import Path

r = subprocess.run(['find', '.', '-name', 'best.pt'], capture_output=True, text=True)
found = [f for f in r.stdout.strip().split('\n') if f]

if found:
    for f in found:
        size = Path(f).stat().st_size / 1e6
        print(f'{f}  ({size:.1f} MB)')
    MODEL_PATH = found[0]
else:
    print('best.pt introuvable - verifie que la cellule 6 a reussi')
    MODEL_PATH = None

In [ ]:
# 8. Test rapide
import glob
from pathlib import Path
from ultralytics import YOLO

if MODEL_PATH:
    m = YOLO(MODEL_PATH)
    print('Classes:', list(m.names.values()))
    for img in glob.glob('data/val/images/*')[:3]:
        r = m(img, conf=0.3, verbose=False)[0]
        dets = [(m.names[int(b.cls)], f'{float(b.conf):.0%}') for b in r.boxes]
        print(f'  {Path(img).name}: {dets if dets else "rien detecte"}')
else:
    print('Pas de modele a tester.')

In [ ]:
# 9. Telechargement de best.pt
from google.colab import files
from pathlib import Path

if MODEL_PATH and Path(MODEL_PATH).exists():
    files.download(MODEL_PATH)
    print('Place best.pt dans: PickAndGo/models/pick_and_go/weights/best.pt')
else:
    print('Erreur: relance la cellule 6 puis la cellule 7.')